# Mammo-FM demo run - preprocess and train in one session

**Project:** Q-INTERVAL-Lite+  |  **Author:** Pasindu Pahasara (104348348)

One Colab session: extract the collection, generate the model inputs, train the
Mammo-FM three-class classifier, and write the bundle the backend loads. Nothing large
is written to or read from Drive, because a round trip through Drive costs more than an
hour and buys durability that a same-day run does not need.

**Input size is 760x456, not 1520x912.** Native sizes in this collection are median
359x207, so 1520x912 upsampled 4.2x - about four times the compute for interpolated
pixels rather than detail. 760x456 is still roughly twice the native median and fits
several times more epochs into the same window. The number lives in one place,
`preprocessing.py`, and everything else reads it from there.

**Run it twice.** First with `SMOKE = True`: about two minutes, exercises the entire
path including the bundle save and reload, then stops. Only when that passes, set
`SMOKE = False` and run all again.

**What it will not do.** No random encoder fallback, no test results feeding any choice,
no invented split for a patient the manifest does not cover, and no describing a
time-limited run as a converged one.

## 1. Setup and configuration

In [8]:
!pip install -q efficientnet_pytorch scikit-learn tqdm
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import os, io, sys, json, time, math, shutil, hashlib, zipfile, random, platform
import subprocess, tempfile, itertools
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from efficientnet_pytorch import EfficientNet
from sklearn.metrics import (f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix, precision_recall_fscore_support)
from tqdm.notebook import tqdm
from google.colab import drive

drive.mount('/content/drive')

# ============================= EDIT THIS CELL ONLY =============================
DRIVE_ROOT = '/content/drive/MyDrive/mamobench-dataset'
ZIP_PATH   = f'{DRIVE_ROOT}/dataset.zip'
SPLIT_PATH = f'{DRIVE_ROOT}/sprint4_audit/frozen_split.json'
AUX_SPLIT  = f'{DRIVE_ROOT}/runs_allsources'      # calibration / new-patient manifests
FM_PATH    = f'{DRIVE_ROOT}/mammo_fm/Mammo-FM_BatmanlabTrained_CLIP.tar'
OUT_ROOT   = f'{DRIVE_ROOT}/runs_demo_v1'         # only small files go here

TIME_BUDGET_HOURS = 4.0      # total for BOTH training stages. Measured, then enforced.
PROBE_SHARE       = 0.35     # of the budget spent on the frozen-encoder probe
SMOKE             = False     # True first: ~2 min, proves the whole path, then stops

SEED            = 42
EFFECTIVE_BATCH = 32
MICRO_BATCH     = 0          # 0 = measure the largest that fits on this GPU
PROBE_HEAD_LR   = 1e-3
FT_HEAD_LR      = 1e-4
FT_ENCODER_LR   = 1e-5
WEIGHT_DECAY    = 1e-4
GRAD_CLIP       = 1.0
MAX_EPOCHS      = 20
PATIENCE        = 5
WARMUP_FRAC     = 0.05
FINAL_FRAC      = 0.01
CALIB_FRACTION  = 0.30
EXCLUDE_FILES   = ['ddsm_4921.jpg']
N_WORKERS       = max(4, (os.cpu_count() or 8))
# ===============================================================================

# Existing settings carried forward. Starting settings, not a claim of optimality.

LOCAL     = '/content/mammo_demo'
RAW_DIR   = f'{LOCAL}/raw'                 # extracted source collection
BASE_DIR  = f'{LOCAL}/baseline'            # generated model inputs
REPORTS   = f'{LOCAL}/reports'
PKG_DIR   = '/content/pkg/classical_session_analysis'
RUN_DIR   = f'{OUT_ROOT}/{"smoke" if SMOKE else "full"}/seed_{SEED}'
for d in (LOCAL, RAW_DIR, BASE_DIR, REPORTS, RUN_DIR):
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = device.type == 'cuda'
BF16 = USE_AMP and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
USE_SCALER = USE_AMP and not BF16
torch.backends.cudnn.benchmark = True
if SMOKE:
    MAX_EPOCHS, PATIENCE, TIME_BUDGET_HOURS = 1, 1, 0.1

CALIB_SEED, VAL_SPLIT_SEED = 20260909, 20260910
PKG = {'python': platform.python_version(), 'torch': torch.__version__,
       'numpy': np.__version__, 'opencv': cv2.__version__, 'pandas': pd.__version__}
T_START = time.time()

def sha256_file(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for b in iter(lambda: fh.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

missing = [p for p in (ZIP_PATH, FM_PATH) if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('not on Drive: ' + ', '.join(missing))
if not os.path.exists(SPLIT_PATH):
    print(f'WARNING: no split manifest at {SPLIT_PATH}. Patients without an assignment '
          'are excluded rather than given an invented one.')

print(f'device {device} | {torch.cuda.get_device_name(0) if device.type=="cuda" else ""} '
      f'| precision {"bf16" if BF16 else ("fp16+scaler" if USE_AMP else "fp32")}')
print(f'{"SMOKE" if SMOKE else "FULL RUN"} | seed {SEED} | budget {TIME_BUDGET_HOURS}h '
      f'| workers {N_WORKERS}')
print(PKG)

NVIDIA A100-SXM4-80GB, 81920 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device cuda | NVIDIA A100-SXM4-80GB | precision bf16
FULL RUN | seed 42 | budget 4.0h | workers 12
{'python': '3.13.15', 'torch': '2.11.0+cu128', 'numpy': '2.1.3', 'opencv': '5.0.0', 'pandas': '2.2.3'}


## 2. The preprocessing contract

`preprocessing.py` is written here and is the only definition of the input pipeline. The
bundle records its SHA-256 and the backend refuses to load a model whose hash disagrees,
so the served pipeline cannot drift from the trained one. Copy **this** file into the
repo next to the bundle.

In [9]:
import os
os.makedirs('/content/pkg/classical_session_analysis/models', exist_ok=True)
PREP_SRC = '/content/pkg/classical_session_analysis/preprocessing.py'
PREP_CODE = r'''"""Baseline preprocessing for the Mammo-FM classical three-class engine.

SINGLE SOURCE OF TRUTH. The training notebook writes this exact file, records its
SHA-256 in the model bundle, and the backend verifies that hash at load time. If the
two ever diverge, loading fails loudly instead of serving predictions produced by a
different pipeline from the one the model was trained on.

Two entry points, deliberately separate so preprocessing cannot be applied twice:

    prepare_raw(image_bytes)        an ordinary uploaded mammogram: crop, resize, pad
    prepare_baseline(image_bytes)   an already-exported prep-v1 baseline PNG: decode only

Both return (tensor_input, geometry). `tensor_input` is float32 (3, H, W), normalised
once. `geometry` records every transform so an attribution map can be mapped back onto
the original pixels.

Depends only on numpy and OpenCV. No torch, no Colab.
"""

from __future__ import annotations

from typing import Any, Dict, Optional, Tuple

import cv2
import numpy as np

__all__ = ['CONTRACT', 'CLASS_NAMES', 'prepare_raw', 'prepare_baseline',
           'decode_bytes', 'detect_breast_crop', 'fit_and_pad', 'normalise',
           'map_to_original', 'PreprocessingError']

CLASS_NAMES = ('Normal', 'Benign', 'Malignant')

# Everything the model input depends on. Copied into the bundle and compared at load.
CONTRACT: Dict[str, Any] = {
    'name': 'mammofm-classical-baseline',
    'prep_version': 'prep-v1',
    # Native image sizes in Mammo-Bench are median 359x207. 1520x912 upsampled 4.2x,
    # which costs ~4x the compute and adds interpolation rather than detail. 760x456 is
    # still ~2x the native median. Change these two numbers and EVERYTHING follows: the
    # training notebook reads the target from here, and the bundle records this file's
    # hash, so the backend cannot silently disagree with the model.
    'target_h': 760,
    'target_w': 456,
    'channels': 3,                     # grayscale replicated, not a colour image
    'norm_mean': [0.485, 0.456, 0.406],
    'norm_std': [0.229, 0.224, 0.225],
    'pad_value': 0,
    'interpolation': 'INTER_AREA when shrinking, INTER_LINEAR when enlarging',
    'crop': {
        'algo': 'conservative-v1',
        'bg_percentile': 1.0, 'abs_margin': 0.02, 'rel_frac': 0.05,
        'margin_frac': 0.03, 'min_area_frac': 0.02, 'rival_ratio': 0.50,
        'faint_factor': 0.50, 'faint_tol': 0.005, 'tight_frac': 0.98,
        'detect_max_side': 1024,
    },
    'clahe': False,                    # the baseline variant carries no CLAHE
    'orientation': 'stored orientation retained; no flips are applied',
    'supported_input_formats': ['PNG', 'JPEG'],
    'unsupported': ['DICOM'],          # no DICOM decoder is implemented or tested here
}

_C = CONTRACT['crop']


class PreprocessingError(ValueError):
    """An input cannot be decoded or does not match the contract.

    ValueError on purpose: router_factory turns ValueError into HTTP 400 with the
    message passed through to the caller, which is right for a bad upload. Anything
    that is not the caller's fault stays a RuntimeError and becomes a 500.
    """


# ---------------------------------------------------------------- decoding ---
def decode_bytes(data: bytes) -> np.ndarray:
    """Decode PNG/JPEG bytes to a single-channel array at its stored bit depth.

    IMREAD_UNCHANGED, so a 16-bit PNG keeps 16 bits. An RGB frame whose channels are
    identical is reduced to one channel; a genuinely coloured image is refused rather
    than silently converted, because that is a sign the wrong file was uploaded.
    """
    if not data:
        raise PreprocessingError('empty image payload')
    buf = np.frombuffer(data, dtype=np.uint8)
    a = cv2.imdecode(buf, cv2.IMREAD_UNCHANGED)
    if a is None:
        raise PreprocessingError(
            'could not decode the image. Supported: %s. DICOM is NOT supported by this '
            'engine - no DICOM decoder is implemented or tested here.'
            % ', '.join(CONTRACT['supported_input_formats']))
    if a.ndim == 3:
        planes = [a[:, :, i] for i in range(min(3, a.shape[2]))]
        if all(np.array_equal(planes[0], p) for p in planes[1:]):
            return planes[0]
        raise PreprocessingError('genuinely coloured image - refused rather than '
                                 'converted, because a mammogram should be grayscale')
    return a


def to_unit(a: np.ndarray) -> np.ndarray:
    """Stored values to [0,1], for crop detection only. Never saved."""
    full = 65535.0 if a.dtype == np.uint16 else 255.0
    return a.astype(np.float32) / full


def to_8bit(a: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
    """8-bit values are preserved. 16-bit is mapped from the smallest container the
    data actually fits, never assuming the full 0-65535 range is used."""
    if a.dtype == np.uint8:
        return a, {'kind': 'identity', 'container': 255, 'scale': 1.0}
    vmax = int(a.max())
    container = next((c for c in (255, 1023, 4095, 16383, 65535) if vmax <= c), 65535)
    scale = 255.0 / container
    return (np.clip(a.astype(np.float32) * scale, 0, 255).round().astype(np.uint8),
            {'kind': 'linear_container', 'container': container, 'scale': scale})


# ------------------------------------------------------------------- crop ---
def detect_breast_crop(gray01: np.ndarray) -> Dict[str, Any]:
    """Conservative breast box on a [0,1] image, exclusive upper bounds.

    Identical in behaviour to the detector used to build the prep-v1 baseline images.
    Faint tissue joined to the main region is always enclosed whatever its size; faint
    signal detached from it is judged by area and, when there is too much, the original
    frame is kept and the status is 'review_required'. Anything uncertain returns the
    full frame rather than a silent crop.
    """
    h, w = gray01.shape
    out = {'box': (0, h, 0, w), 'status': 'review_required', 'reason': '',
           'retained_frac': 1.0, 'connected_faint_added': False, 'margin_applied': [0, 0]}
    try:
        if h < 8 or w < 8:
            out['reason'] = 'image too small to analyse'
            return out
        bg = float(np.percentile(gray01, _C['bg_percentile']))
        bright = float(np.percentile(gray01, 99.0))
        thr = bg + max(_C['abs_margin'], _C['rel_frac'] * max(bright - bg, 1e-6))
        faint_thr = bg + _C['faint_factor'] * (thr - bg)

        mask = (gray01 > thr).astype(np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
        if mask.sum() < _C['min_area_frac'] * h * w:
            out['reason'] = 'almost no foreground above the background level'
            return out

        n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        comps = []
        for i in range(1, n):
            x, y, bw, bh, area = stats[i]
            if area < _C['min_area_frac'] * h * w * 0.25:
                continue
            if bw > 0.95 * w and bh > 0.95 * h and area / max(bw * bh, 1) < 0.30:
                continue                       # a bright rim around the whole picture
            comps.append({'i': i, 'x': int(x), 'y': int(y), 'w': int(bw), 'h': int(bh),
                          'area': int(area)})
        if not comps:
            out['reason'] = 'no component survived the frame-border and size filters'
            return out
        comps.sort(key=lambda c: -c['area'])
        main = comps[0]
        if len(comps) > 1 and comps[1]['area'] / main['area'] > _C['rival_ratio']:
            out['reason'] = 'two components of similar size - cannot tell which is breast'
            return out
        if main['area'] < _C['min_area_frac'] * h * w:
            out['reason'] = 'largest component is too small'
            return out

        y0, y1 = main['y'], main['y'] + main['h']
        x0, x1 = main['x'], main['x'] + main['w']

        faint = (gray01 > faint_thr).astype(np.uint8)
        faint = cv2.morphologyEx(faint, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
        _, flab, fstats, _ = cv2.connectedComponentsWithStats(faint, connectivity=8)
        touching = set(np.unique(flab[labels == main['i']])) - {0}
        for lbl in touching:                   # connected faint tissue is always kept
            fx, fy, fw, fh, _a = fstats[lbl]
            if fy < y0 or fy + fh > y1 or fx < x0 or fx + fw > x1:
                out['connected_faint_added'] = True
            y0, y1 = min(y0, int(fy)), max(y1, int(fy + fh))
            x0, x1 = min(x0, int(fx)), max(x1, int(fx + fw))

        loose = faint.astype(bool).copy()
        for lbl in touching:
            loose[flab == lbl] = False
        loose[y0:y1, x0:x1] = False
        if float(loose.sum()) / (h * w) > _C['faint_tol']:
            out['reason'] = 'faint signal detached from the main region - frame kept'
            return out

        my = int(round(_C['margin_frac'] * (y1 - y0)))    # margin AFTER the boundary
        mx = int(round(_C['margin_frac'] * (x1 - x0)))
        y0, y1 = max(0, y0 - my), min(h, y1 + my)
        x0, x1 = max(0, x0 - mx), min(w, x1 + mx)

        retained = (y1 - y0) * (x1 - x0) / float(h * w)
        out.update(box=(int(y0), int(y1), int(x0), int(x1)),
                   retained_frac=round(retained, 4), margin_applied=[my, mx],
                   status='already_tight' if retained >= _C['tight_frac'] else 'cropped',
                   reason='background margin removed')
        return out
    except Exception as exc:                   # never lose an image to a crop failure
        out['reason'] = f'crop detection failed: {type(exc).__name__}: {exc}'
        return out


def crop_box(raw: np.ndarray) -> Dict[str, Any]:
    """Detect on a downsample when the image is large, then map the box back to the
    original grid, rounding outward so the mapping can only ever keep more tissue."""
    g = to_unit(raw)
    h, w = g.shape
    f = min(1.0, _C['detect_max_side'] / float(max(h, w)))
    if f >= 1.0:
        info = detect_breast_crop(g)
        info['detect_scale'] = 1.0
        return info
    small = cv2.resize(g, (max(1, int(round(w * f))), max(1, int(round(h * f)))),
                       interpolation=cv2.INTER_AREA)
    info = detect_breast_crop(small)
    sy, sx = h / float(small.shape[0]), w / float(small.shape[1])
    y0, y1, x0, x1 = info['box']
    info['box'] = (max(0, int(np.floor(y0 * sy))), min(h, int(np.ceil(y1 * sy))),
                   max(0, int(np.floor(x0 * sx))), min(w, int(np.ceil(x1 * sx))))
    info['detect_scale'] = round(f, 4)
    return info


# ------------------------------------------------------- resize and pad ---
def fit_and_pad(img8: np.ndarray) -> Tuple[np.ndarray, Dict[str, Any]]:
    """One proportional resize into the target, then centred black padding."""
    H, W = CONTRACT['target_h'], CONTRACT['target_w']
    h, w = img8.shape
    s = min(H / float(h), W / float(w))
    nh = min(H, max(1, int(round(h * s))))
    nw = min(W, max(1, int(round(w * s))))
    interp = cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR
    small = cv2.resize(img8, (nw, nh), interpolation=interp)
    top, left = (H - nh) // 2, (W - nw) // 2
    out = cv2.copyMakeBorder(small, top, H - nh - top, left, W - nw - left,
                             cv2.BORDER_CONSTANT, value=CONTRACT['pad_value'])
    if out.shape != (H, W):
        raise PreprocessingError(f'padded to {out.shape}, expected {(H, W)}')
    return out, {'scale': float(s), 'resized_h': nh, 'resized_w': nw,
                 'pad_top': top, 'pad_bottom': H - nh - top,
                 'pad_left': left, 'pad_right': W - nw - left,
                 'interpolation': 'area' if s < 1 else 'linear'}


def normalise(img8: np.ndarray) -> np.ndarray:
    """Grayscale uint8 -> float32 (3, H, W), replicated and normalised ONCE."""
    x = img8.astype(np.float32) / 255.0
    x = np.repeat(x[None, :, :], CONTRACT['channels'], axis=0)
    mean = np.asarray(CONTRACT['norm_mean'], np.float32)[:, None, None]
    std = np.asarray(CONTRACT['norm_std'], np.float32)[:, None, None]
    return (x - mean) / std


# --------------------------------------------------------- entry points ---
def prepare_raw(data: bytes) -> Tuple[np.ndarray, Dict[str, Any]]:
    """An ordinary uploaded mammogram. Crop, resize, pad, normalise - once."""
    raw = decode_bytes(data)
    info = crop_box(raw)
    y0, y1, x0, x1 = info['box']
    crop = raw[y0:y1, x0:x1]
    img8, conv = to_8bit(crop)
    padded, geom = fit_and_pad(img8)
    geom.update(entry='raw', orig_h=int(raw.shape[0]), orig_w=int(raw.shape[1]),
                crop_y0=y0, crop_y1=y1, crop_x0=x0, crop_x1=x1,
                crop_h=int(crop.shape[0]), crop_w=int(crop.shape[1]),
                crop_status=info['status'], crop_reason=info['reason'],
                to8=conv, padded_8bit=padded)
    return normalise(padded), geom


def prepare_baseline(data: bytes) -> Tuple[np.ndarray, Dict[str, Any]]:
    """An already-exported prep-v1 baseline image. Decode and normalise only.

    Cropping, resizing and padding have already been applied to these files. Doing any
    of it again would be a second pass over an image that is already at the target.
    """
    raw = decode_bytes(data)
    H, W = CONTRACT['target_h'], CONTRACT['target_w']
    if raw.shape != (H, W):
        raise PreprocessingError(
            f'baseline entry point expects an image already at {H}x{W}, got '
            f'{raw.shape}. Use prepare_raw() for an unprocessed mammogram.')
    if raw.dtype != np.uint8:
        raise PreprocessingError(f'baseline images are 8-bit; got {raw.dtype}')
    geom = {'entry': 'baseline', 'orig_h': H, 'orig_w': W, 'scale': 1.0,
            'resized_h': H, 'resized_w': W, 'pad_top': 0, 'pad_bottom': 0,
            'pad_left': 0, 'pad_right': 0, 'crop_y0': 0, 'crop_y1': H,
            'crop_x0': 0, 'crop_x1': W, 'crop_h': H, 'crop_w': W,
            'interpolation': 'none', 'padded_8bit': raw}
    return normalise(raw), geom


def map_to_original(heat: np.ndarray, geom: Dict[str, Any]) -> np.ndarray:
    """Take a (target_h, target_w) attribution map back to the ORIGINAL image.

    Reverses padding, then the resize, then places the result at the recorded crop
    coordinates on a full-size canvas. A cropped heatmap is never pasted straight onto
    a differently shaped original.
    """
    H, W = CONTRACT['target_h'], CONTRACT['target_w']
    if heat.shape != (H, W):
        raise PreprocessingError(f'attribution map is {heat.shape}, expected {(H, W)}')
    t, l = int(geom['pad_top']), int(geom['pad_left'])
    nh, nw = int(geom['resized_h']), int(geom['resized_w'])
    inner = heat[t:t + nh, l:l + nw]                       # undo the padding
    ch, cw = int(geom['crop_h']), int(geom['crop_w'])
    inner = cv2.resize(inner, (cw, ch), interpolation=cv2.INTER_LINEAR)   # undo resize
    canvas = np.zeros((int(geom['orig_h']), int(geom['orig_w'])), dtype=heat.dtype)
    canvas[int(geom['crop_y0']):int(geom['crop_y1']),
           int(geom['crop_x0']):int(geom['crop_x1'])] = inner              # crop origin
    return canvas
'''
open(PREP_SRC, 'w').write(PREP_CODE)
open('/content/pkg/classical_session_analysis/__init__.py', 'w').write('')

import sys, hashlib
sys.path.insert(0, '/content/pkg')
from classical_session_analysis import preprocessing as PREP
PREP_SHA = hashlib.sha256(open(PREP_SRC, 'rb').read()).hexdigest()
TARGET_H, TARGET_W = PREP.CONTRACT['target_h'], PREP.CONTRACT['target_w']
CLASSES = list(PREP.CLASS_NAMES)
print(f'preprocessing.py written to {PREP_SRC}')
print(f'  sha256 {PREP_SHA[:16]}  target {TARGET_H}x{TARGET_W}  '
      f'classes {CLASSES}')
print('This file is the ONLY definition of the input pipeline. The bundle records\n'
      'its hash and the backend refuses to load a model whose hash disagrees, so\n'
      'copy THIS file into the repo alongside the bundle.')

preprocessing.py written to /content/pkg/classical_session_analysis/preprocessing.py
  sha256 5adad4f338d3f1a4  target 760x456  classes ['Normal', 'Benign', 'Malignant']
This file is the ONLY definition of the input pipeline. The bundle records
its hash and the backend refuses to load a model whose hash disagrees, so
copy THIS file into the repo alongside the bundle.


## 3. Data: extract, label, split

Labels are the existing mapping - exact match after stripping to Normal, Benign,
Malignant, with Suspicious Malignant set aside rather than relabelled. Patient
assignments are preserved from the frozen split manifest and never re-derived; a patient
the manifests do not cover is excluded rather than given an invented partition.

In [10]:
# Extract the source collection locally. Nothing large is read from or written to Drive.
marker = f'{LOCAL}/.extracted.json'
zid = {'bytes': os.path.getsize(ZIP_PATH)}
if os.path.exists(marker) and json.load(open(marker)) == zid:
    print('already extracted')
else:
    t0 = time.time()
    local_zip = f'{LOCAL}/dataset.zip'
    if not (os.path.exists(local_zip) and os.path.getsize(local_zip) == zid['bytes']):
        shutil.copyfile(ZIP_PATH, local_zip)
        print(f'copied {zid["bytes"]/1e9:.2f} GB from Drive in {time.time()-t0:.0f}s')
    with zipfile.ZipFile(local_zip) as zf:
        for i in zf.infolist():
            n = i.filename.replace('\\', '/')
            if n.startswith('/') or '..' in n.split('/'):
                raise RuntimeError(f'unsafe archive member {n}')
        zf.extractall(RAW_DIR)
    json.dump(zid, open(marker, 'w'))
    print(f'extracted in {time.time()-t0:.0f}s')

cand = [os.path.join(r, 'metadata.csv') for r, _, fs in os.walk(RAW_DIR) if 'metadata.csv' in fs]
if not cand:
    raise FileNotFoundError(f'no metadata.csv under {RAW_DIR}')
META_PATH = min(cand, key=lambda p: p.count(os.sep))
IMG_ROOT = os.path.dirname(META_PATH)
print(f'metadata {META_PATH}')

meta = pd.read_csv(META_PATH, dtype={'source_subjectID': str, 'source_dataset': str,
                                     'preprocessed_image_path': str, 'classification': str},
                   low_memory=False)
df = meta.copy()
resolved = df.classification.str.strip().map({c: c for c in CLASSES})
df['label_text'] = resolved
df['label'] = resolved.map({c: i for i, c in enumerate(CLASSES)})
df['eligible'] = resolved.notna()
df['rel_path'] = df.preprocessed_image_path.astype(str).str.replace('\\', '/', regex=False)
df['image_path'] = IMG_ROOT + '/' + df.rel_path
df['image_id'] = [hashlib.sha256(p.encode()).hexdigest()[:16] for p in df.rel_path]
df['basename'] = df.rel_path.str.split('/').str[-1]

# acquisition from the path first, filename second; unmatched stays unknown
def acq(r):
    if str(r['source_dataset']).lower() != 'cdd-cesm':
        return 'standard'
    p = str(r.get('original_source_path', '') or '').lower().replace('\\', '/')
    n = str(r['rel_path']).lower()
    for tok, v in (('low_energy', 'cdd_low_energy'), ('subtracted', 'cdd_recombined'),
                   ('recombined', 'cdd_recombined')):
        if tok in p:
            return v
    for tok, v in (('_dm', 'cdd_low_energy'), ('_cm', 'cdd_recombined')):
        if tok in n:
            return v
    return 'cdd_unknown'
df['acquisition'] = [acq(r) for r in df.to_dict('records')]

# patient key: source-prefixed only where a raw id appears under more than one source
multi = df.groupby('source_subjectID').source_dataset.nunique()
amb = set(multi[multi > 1].index)
df['patient'] = np.where(df.source_subjectID.isin(amb),
                         df.source_dataset.str.lower() + ':' + df.source_subjectID,
                         df.source_subjectID)
raw_to_keys = {}
for raw, key in zip(df.source_subjectID, df.patient):
    raw_to_keys.setdefault(raw, set()).add(key)
def to_keys(ids):
    out = set()
    for x in ids:
        out |= raw_to_keys.get(str(x), {str(x)})
    return out

hist = json.load(open(SPLIT_PATH)) if os.path.exists(SPLIT_PATH) else {}
hist = hist.get('splits', hist)
splits = {k.lower().replace('validation', 'val'): to_keys(v)
          for k, v in hist.items() if isinstance(v, list)}
for fname in ('derived_validation_split.json', 'new_patient_allocation.json'):
    p = f'{AUX_SPLIT}/{fname}'
    if not os.path.exists(p):
        continue
    data = json.load(open(p))
    if 'derived' in fname:
        v = to_keys(data.get('val', []))
        splits['val'] = splits.get('val', set()) | v
        splits['train'] = splits.get('train', set()) - v
    else:
        for k in ('train', 'val', 'test'):
            splits[k] = splits.get(k, set()) | to_keys(data.get(k, []))
for a, b in itertools.combinations(splits, 2):
    assert not (splits[a] & splits[b]), f'patients in both {a} and {b}'
df['split'] = df.patient.map({p: k for k, v in splits.items() for p in v}).fillna('unassigned')

calib = set()
cp = f'{AUX_SPLIT}/calibration_split.json'
if os.path.exists(cp):
    calib = to_keys(json.load(open(cp)).get('calibration', []))
elif 'val' in splits:      # reserve one, with its own fixed seed, and save it
    vp = sorted(splits['val'])
    calib = set(np.random.default_rng(CALIB_SEED).choice(
        vp, int(round(CALIB_FRACTION * len(vp))), replace=False).tolist())
    os.makedirs(AUX_SPLIT, exist_ok=True)
    json.dump({'calibration': sorted(calib), 'seed': CALIB_SEED}, open(cp, 'w'), indent=2)
    print(f'reserved {len(calib)} calibration patients -> {cp}')
df['role'] = np.where(df.patient.isin(calib) & (df.split == 'val'), 'calibration', df.split)

print(f'\n{len(df)} rows | {int(df.eligible.sum())} eligible | '
      f'{df.patient.nunique()} patients')
print(pd.crosstab(df.role, df.label_text).to_string())

already extracted
metadata /content/mammo_demo/raw/dataset/metadata.csv

19731 rows | 19496 eligible | 5860 patients
label_text   Benign  Malignant  Normal
role                                  
calibration     294        357     270
selection       603        803     554
test            903       1269     772
train          3647       4884    3118
val             622        871     529


## 4. Generate the model inputs, then check them

Baseline variant only - no masters, no CLAHE. Conservative crop, one proportional resize,
centred padding, written atomically. Then exclusions with reasons, and duplicate content
spanning partitions resolved by dropping the copy in the *less* protected partition, so
nothing held out moves into training.

In [11]:
# Generate the model inputs once, locally, using the SAME preprocessing.py the backend
# ships. Baseline variant only - no masters, no CLAHE - because only the baseline is
# needed to train and the other two are pure cost today.
work = df[df.eligible].copy()
print(f'preprocessing {len(work)} eligible images to {TARGET_H}x{TARGET_W} ...')

def make_one(r):
    rec = {'image_id': r['image_id'], 'status': 'failed', 'reason': '',
           'baseline_path': '', 'pixel_sha': ''}
    try:
        with open(r['image_path'], 'rb') as fh:
            data = fh.read()
        raw = PREP.decode_bytes(data)
        info = PREP.crop_box(raw)
        y0, y1, x0, x1 = info['box']
        crop = raw[y0:y1, x0:x1]
        img8, conv = PREP.to_8bit(crop)
        padded, geom = PREP.fit_and_pad(img8)
        out = f'{BASE_DIR}/{r["source_dataset"]}/{r["image_id"]}.png'
        os.makedirs(os.path.dirname(out), exist_ok=True)
        ok, buf = cv2.imencode('.png', padded, [cv2.IMWRITE_PNG_COMPRESSION, 1])
        if not ok:
            raise RuntimeError('png encode failed')
        tmp = out + '.tmp'
        with open(tmp, 'wb') as fh:
            fh.write(buf.tobytes())
        os.replace(tmp, out)                       # atomic: no half-written input
        rec.update(status='review_required' if info['status'] == 'review_required'
                   else 'processed',
                   reason=info['reason'], baseline_path=out,
                   crop_status=info['status'], retained_frac=info['retained_frac'],
                   annotation_overlap=info.get('annotation_overlap', False),
                   orig_h=int(raw.shape[0]), orig_w=int(raw.shape[1]),
                   crop_h=int(crop.shape[0]), crop_w=int(crop.shape[1]),
                   resize_scale=round(float(geom['scale']), 5),
                   pad_frac=round(1 - (geom['resized_h'] * geom['resized_w'])
                                  / float(TARGET_H * TARGET_W), 4),
                   to8_container=conv['container'],
                   pixel_sha=hashlib.sha256(np.ascontiguousarray(raw).tobytes()
                                            ).hexdigest()[:32])
    except Exception as exc:
        rec['reason'] = f'{type(exc).__name__}: {exc}'
    return rec

t0 = time.time()
rows = work.to_dict('records')
with ThreadPoolExecutor(N_WORKERS) as ex:
    made = pd.DataFrame(list(tqdm(ex.map(make_one, rows), total=len(rows), desc='preprocess')))
dt = time.time() - t0
print(f'{len(made)} images in {dt/60:.1f} min ({len(made)/max(dt,1):.0f}/s)')
print(made.status.value_counts().to_string())

man = work.merge(made, on='image_id', how='left')
man.to_csv(f'{REPORTS}/manifest.csv', index=False)

# ---- exclusions, each with a reason -------------------------------------------------
excl = []
def drop(mask, reason):
    for iid in man.loc[mask.fillna(False), 'image_id']:
        excl.append({'image_id': iid, 'reason': reason})

drop(man.status.eq('failed'), 'preprocessing failure')
drop(man.status.eq('review_required'), 'unresolved crop review - kept its full frame')
drop(man.get('annotation_overlap', pd.Series(False, index=man.index)).fillna(False).astype(bool),
     'flagged: marker or annotation overlapping tissue')
drop(man.basename.isin(EXCLUDE_FILES), 'explicitly flagged for inspection')
drop(man.split.eq('unassigned'), 'no partition in the frozen split manifest')

# identical image content spanning partitions: the copy in the MORE protected partition
# is kept, the others are excluded. Nothing is moved, so no held-out case enters training.
PROT = {'test': 0, 'calibration': 1, 'val': 2, 'train': 3}
ok_rows = man[man.status.isin(['processed', 'review_required']) & man.pixel_sha.ne('')]
dupe_groups = dupe_rows = 0
for sha, g in ok_rows.groupby('pixel_sha'):
    if len(g) < 2 or g.role.nunique() < 2:
        continue
    dupe_groups += 1
    keep = g.sort_values(['role', 'image_id'], key=lambda s: s.map(PROT).fillna(9)
                         if s.name == 'role' else s).image_id.iloc[0]
    for iid in g.image_id:
        if iid != keep:
            excl.append({'image_id': iid, 'reason': 'identical image content also present '
                         'in a more protected partition; that copy is the one kept'})
            dupe_rows += 1
print(f'duplicate-content groups spanning partitions: {dupe_groups} '
      f'({dupe_rows} images excluded, 0 moved)')

EXCLUSIONS = (pd.DataFrame(excl).drop_duplicates('image_id') if excl
              else pd.DataFrame(columns=['image_id', 'reason']))
EXCLUSIONS.to_csv(f'{REPORTS}/exclusion_manifest.csv', index=False)
data = man[~man.image_id.isin(set(EXCLUSIONS.image_id))
           & man.status.eq('processed')].copy()
data['path'] = data.baseline_path

print(f'\n{len(EXCLUSIONS)} excluded:')
print(EXCLUSIONS.reason.str.slice(0, 60).value_counts().to_string())

# every partition must still be disjoint by patient and by content
assert (data.groupby('patient').role.nunique() <= 1).all(), 'a patient spans partitions'
assert (data.groupby('pixel_sha').role.nunique() <= 1).all(), 'content spans partitions'
print('verified: no patient and no identical image content spans a partition')

train_df = data[data.role == 'train']
val_df = data[data.role == 'val']
calib_df = data[data.role == 'calibration']
test_df = data[data.role == 'test']
if SMOKE:
    train_df = train_df.groupby('label', group_keys=False).head(24)
    val_df = val_df.groupby('label', group_keys=False).head(12)

print('\nimages by partition and class')
print(pd.crosstab(data.role, data.label_text).to_string())
print('\nimages by source and partition')
print(pd.crosstab(data.source_dataset, data.role).to_string())
print('\npatients by partition')
print(data.groupby('role').patient.nunique().to_string())
print('\nacquisition kept:', data.acquisition.value_counts().to_dict())
print(f'\nresize scale median {data.resize_scale.median():.2f}x '
      f'({int((data.resize_scale > 1).mean()*100)}% enlarged) | '
      f'padding median {data.pad_frac.median():.3f}')
print(f'label mapping: ' + ', '.join(f'{c}={i}' for i, c in enumerate(CLASSES)))
print(f'training on {len(train_df)} images / {train_df.patient.nunique()} patients'
      + ('   (SMOKE SUBSET)' if SMOKE else ''))

preprocessing 19496 eligible images to 760x456 ...


preprocess:   0%|          | 0/19496 [00:00<?, ?it/s]

19496 images in 3.7 min (87/s)
status
processed          19460
review_required       36
duplicate-content groups spanning partitions: 4 (4 images excluded, 0 moved)

41 excluded:
reason
unresolved crop review - kept its full frame                    36
identical image content also present in a more protected par     4
explicitly flagged for inspection                                1
verified: no patient and no identical image content spans a partition

images by partition and class
label_text   Benign  Malignant  Normal
role                                  
calibration     294        356     270
selection       602        799     554
test            901       1265     772
train          3639       4873    3112
val             618        871     529

images by source and partition
role            calibration  selection  test  train   val
source_dataset                                           
cdd-cesm                 41          0   160    679   121
cmmd                    238      

## 5. Loader, encoder, model, and the largest batch that fits

Every B5 parameter must be present in the checkpoint with a matching shape. The
micro-batch is measured on this GPU rather than guessed, by pushing real forward and
backward passes until one runs out of memory.

In [12]:
class BaselineDS(Dataset):
    """The generated inputs are already at the target. Decode and normalise once."""
    def __init__(self, frame):
        self.paths = frame.path.tolist(); self.labels = frame.label.astype(int).tolist()
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with open(self.paths[i], 'rb') as fh:
            x, _ = PREP.prepare_baseline(fh.read())
        return torch.from_numpy(x), self.labels[i], i

def loader(frame, shuffle, bs, persistent=False):
    # persistent workers only for the training loader, which is reused every epoch.
    # evaluate() builds a fresh loader each call; keeping those alive would leak a
    # worker pool per epoch until the runtime runs out of file descriptors.
    nw = min(8, N_WORKERS)
    return DataLoader(BaselineDS(frame), batch_size=bs, shuffle=shuffle, num_workers=nw,
                      pin_memory=(device.type == 'cuda'),
                      persistent_workers=persistent and nw > 0, prefetch_factor=4)

_x, _y, _ = BaselineDS(train_df.head(1))[0]
assert tuple(_x.shape) == (3, TARGET_H, TARGET_W), _x.shape
print(f'sample {tuple(_x.shape)} {_x.dtype} range [{_x.min():.2f}, {_x.max():.2f}] '
      f'label {_y} ({CLASSES[_y]})')

# ---- encoder -------------------------------------------------------------------------
ckpt = torch.load(FM_PATH, map_location='cpu', weights_only=False)
full_sd = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
encoder_state = {k[len('image_encoder.'):]: v for k, v in full_sd.items()
                 if k.startswith('image_encoder.')}
if not encoder_state:
    raise RuntimeError('no image_encoder.* keys in the checkpoint. Refusing to continue '
                       'with a randomly initialised encoder.')
reference = EfficientNet.from_name('efficientnet-b5').state_dict()
ref_keys, ck_keys = set(reference), set(encoder_state)
shape_bad = [k for k in ref_keys & ck_keys
             if tuple(reference[k].shape) != tuple(encoder_state[k].shape)]
missing_bad = [k for k in ref_keys - ck_keys if not k.startswith('_fc.')]
print(f'B5 params {len(ref_keys)} | checkpoint {len(ck_keys)} | '
      f'covered {len(ref_keys & ck_keys)} ({100*len(ref_keys & ck_keys)/len(ref_keys):.1f}%)')
assert not shape_bad, f'shape mismatch {shape_bad[:5]}'
assert not missing_bad, f'{len(missing_bad)} encoder params absent: {missing_bad[:6]}'
print('only the unused _fc.* ImageNet head is absent, which extract_features never calls')
FM_SHA = sha256_file(FM_PATH)

_e = EfficientNet.from_name('efficientnet-b5').eval()
with torch.no_grad():
    FEAT_DIM = int(_e.extract_features(torch.zeros(1, 3, 64, 64)).shape[1])
del _e
print(f'feature dim {FEAT_DIM} | checkpoint sha {FM_SHA[:16]}')

class MammoFM3C(nn.Module):
    def __init__(self, encoder, feat_dim):
        super().__init__()
        self.backbone = encoder
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Linear(feat_dim, 3)
    def feature_map(self, x):
        return self.backbone.extract_features(x)
    def head_from_map(self, f):
        return self.head(self.pool(f).flatten(1))
    def forward(self, x):
        return self.head_from_map(self.feature_map(x))

def build_model():
    enc = EfficientNet.from_name('efficientnet-b5')
    res = enc.load_state_dict({k: v for k, v in encoder_state.items() if k in ref_keys},
                              strict=False)
    assert not [k for k in res.missing_keys if not k.startswith('_fc.')]
    assert not res.unexpected_keys
    m = MammoFM3C(enc, FEAT_DIM)
    nn.init.normal_(m.head.weight, std=0.01)
    c = np.clip(train_df.label.value_counts().sort_index().reindex(range(3), fill_value=0)
                .to_numpy().astype(float), 1, None)
    with torch.no_grad():
        m.head.bias.copy_(torch.tensor(np.log(c / c.sum()), dtype=torch.float32))
    return m.to(device)

def _pin_eval(module):
    module.train = (lambda self, mode=True: nn.Module.train(self, False)).__get__(
        module, module.__class__)
    nn.Module.train(module, False)

def set_stage(model, stage):
    """probe: encoder frozen AND pinned to eval, so model.train() cannot restart its
    BatchNorm running statistics. finetune: encoder trains, BatchNorm statistics stay
    frozen while their affine parameters learn."""
    for m in [model.backbone] + list(model.backbone.modules()):
        m.__dict__.pop('train', None)
    if stage == 'probe':
        for p in model.backbone.parameters():
            p.requires_grad_(False)
        _pin_eval(model.backbone)
    else:
        for p in model.backbone.parameters():
            p.requires_grad_(True)
        model.backbone.train()
        for m in model.backbone.modules():
            if isinstance(m, nn.BatchNorm2d):
                _pin_eval(m)
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def param_groups(model, head_lr, encoder_lr):
    g = {}
    for name, p in model.named_parameters():
        if p.requires_grad:
            g.setdefault(('encoder' if name.startswith('backbone.') else 'head',
                          p.ndim > 1), []).append(p)
    return [{'params': ps, 'lr': encoder_lr if k[0] == 'encoder' else head_lr,
             'weight_decay': WEIGHT_DECAY if k[1] else 0.0} for k, ps in g.items()]

# ---- how large a micro-batch actually fits, measured rather than guessed --------------
if MICRO_BATCH:
    print(f'micro-batch fixed at {MICRO_BATCH}')
else:
    MICRO_BATCH = 1
    probe = build_model(); set_stage(probe, 'finetune')       # the heavier of the two
    opt = AdamW(param_groups(probe, 1e-4, 1e-5))
    for bs in (2, 4, 8, 12, 16, 24, 32):
        try:
            xb = torch.randn(bs, 3, TARGET_H, TARGET_W, device=device)
            yb = torch.zeros(bs, dtype=torch.long, device=device)
            with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                loss = nn.functional.cross_entropy(probe(xb), yb)
            loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
            MICRO_BATCH = bs
            del xb, yb, loss
            torch.cuda.synchronize(); torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            break
    del probe, opt
    torch.cuda.empty_cache()
    print(f'largest micro-batch that fits: {MICRO_BATCH} '
          f'(peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB)')
ACCUM = max(1, EFFECTIVE_BATCH // MICRO_BATCH)
print(f'micro-batch {MICRO_BATCH} x accumulation {ACCUM} = effective '
      f'{MICRO_BATCH * ACCUM} (target {EFFECTIVE_BATCH})')

sample (3, 760, 456) torch.float32 range [-2.12, 2.50] label 0 (Normal)
B5 params 854 | checkpoint 852 | covered 852 (99.8%)
only the unused _fc.* ImageNet head is absent, which extract_features never calls
feature dim 2048 | checkpoint sha 58ae959de52d563a
largest micro-batch that fits: 32 (peak 36.1 GB)
micro-batch 32 x accumulation 1 = effective 32 (target 32)


### 5.1 Training loop

Loss summed and divided by the number of examples actually in each accumulation group,
including the final partial one. The scheduler advances only on updates that really
happened. Selection is validation macro-F1, and each stage stops when the next epoch
would exceed its share of the time budget.

In [13]:
criterion = nn.CrossEntropyLoss(reduction='sum')   # divided by the ACTUAL group size

def lr_lambda(step, total):
    warm = max(1, int(WARMUP_FRAC * total))
    if step < warm:
        return step / warm
    p = (step - warm) / max(1, total - warm)
    return FINAL_FRAC + (1 - FINAL_FRAC) * 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

@torch.no_grad()
def evaluate(model, frame, bs=None):
    model.eval()
    L, Y, I = [], [], []
    for xb, yb, ib in tqdm(loader(frame, False, bs or MICRO_BATCH * 2), leave=False,
                           desc='eval'):
        with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(xb.to(device, non_blocking=True))
        L.append(out.float().cpu()); Y.append(yb); I.append(ib)
    logits = torch.cat(L)
    return torch.softmax(logits, 1).numpy(), torch.cat(Y).numpy(), \
        torch.cat(I).numpy(), logits.numpy()

def metrics(probs, y):
    pred = probs.argmax(1)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=[0, 1, 2],
                                                      zero_division=0)
    out = {'accuracy': float((pred == y).mean()),
           'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
           'per_class': {CLASSES[i]: {'precision': float(pr[i]), 'recall': float(rc[i]),
                                      'f1': float(f1[i]), 'support': int(sup[i])}
                         for i in range(3)},
           'confusion_matrix': confusion_matrix(y, pred, labels=[0, 1, 2]).tolist()}
    mal = (y == 2).astype(int)
    out['malignant_vs_rest'] = (
        {'roc_auc': float(roc_auc_score(mal, probs[:, 2])),
         'average_precision': float(average_precision_score(mal, probs[:, 2])),
         'prevalence': float(mal.mean())}
        if mal.sum() and (1 - mal).sum() else
        {'roc_auc': None, 'average_precision': None, 'note': 'one class only'})
    return out

def train_stage(model, stage, head_lr, enc_lr, tag, budget_s):
    trainable = set_stage(model, stage)
    opt = AdamW(param_groups(model, head_lr, enc_lr))
    scaler = torch.amp.GradScaler('cuda', enabled=USE_SCALER)
    dl = loader(train_df, True, MICRO_BATCH, persistent=True)
    steps = max(1, math.ceil(len(dl) / ACCUM))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: lr_lambda(s, steps * MAX_EPOCHS))
    print(f'[{tag}] {trainable:,} trainable | {steps} steps/epoch | '
          f'budget {budget_s/60:.0f} min')

    best = {'macro_f1': -1.0, 'epoch': -1}
    bad, hist, times, t_stage = 0, [], [], time.time()
    for epoch in range(MAX_EPOCHS):
        t0 = time.time()
        model.train()
        if stage == 'probe':
            model.backbone.eval()
        opt.zero_grad(set_to_none=True)
        gl, gn = 0.0, 0
        pbar = tqdm(dl, desc=f'[{tag}] epoch {epoch+1}', leave=False)
        for bi, (xb, yb, _) in enumerate(pbar):
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                loss_sum = criterion(model(xb), yb)
            if not torch.isfinite(loss_sum):
                raise RuntimeError(f'non-finite loss at batch {bi}')
            (scaler.scale(loss_sum) if scaler.is_enabled() else loss_sum).backward()
            gl += float(loss_sum.detach()); gn += yb.numel()
            if (bi + 1) % ACCUM == 0 or bi == len(dl) - 1:
                for g in opt.param_groups:          # divide by the ACTUAL group size,
                    for p in g['params']:           # including a final partial group
                        if p.grad is not None:
                            p.grad /= gn
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    [p for g in opt.param_groups for p in g['params']], GRAD_CLIP)
                if scaler.is_enabled():
                    s0 = scaler.get_scale(); scaler.step(opt); scaler.update()
                    stepped = scaler.get_scale() >= s0
                else:
                    opt.step(); stepped = True
                if stepped:
                    sched.step()                    # schedule follows real updates only
                opt.zero_grad(set_to_none=True)
                pbar.set_postfix(loss=f'{gl/max(gn,1):.4f}')
                gl, gn = 0.0, 0

        m = metrics(*evaluate(model, val_df)[:2])
        dt = time.time() - t0; times.append(dt)
        hist.append({'epoch': epoch + 1, 'seconds': round(dt, 1),
                     'macro_f1': m['macro_f1'], 'accuracy': m['accuracy'],
                     'mal_auc': m['malignant_vs_rest']['roc_auc']})
        used, mean_dt = time.time() - t_stage, float(np.mean(times))
        left = max(0, budget_s - used)
        print(f"[{tag}] epoch {epoch+1}  {dt/60:.1f} min  val macro-F1 {m['macro_f1']:.4f}"
              f"  acc {m['accuracy']:.4f}  mal-AUC {m['malignant_vs_rest']['roc_auc']}"
              f"  | {left/60:.0f} min budget left, ~{int(left//max(mean_dt,1))} more epochs")

        torch.save({'model': model.state_dict(), 'epoch': epoch, 'stage': stage,
                    'best': best}, f'{RUN_DIR}/{tag}_last.pth')
        if m['macro_f1'] > best['macro_f1']:
            best = {'macro_f1': m['macro_f1'], 'epoch': epoch + 1, 'metrics': m}
            torch.save({'model': model.state_dict(), 'epoch': epoch, 'stage': stage,
                        'best': best}, f'{RUN_DIR}/{tag}_best.pth')
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f'[{tag}] early stop: {bad} epochs without improvement')
                break
        if used + mean_dt > budget_s:
            print(f'[{tag}] stopping: the next epoch would exceed the budget. This is a '
                  'TIME-LIMITED run, not a converged one.')
            break

    json.dump({'history': hist, 'best': best, 'stage': stage},
              open(f'{RUN_DIR}/{tag}_history.json', 'w'), indent=2, default=str)
    model.load_state_dict(torch.load(f'{RUN_DIR}/{tag}_best.pth',
                                     map_location='cpu', weights_only=False)['model'])
    print(f"[{tag}] best val macro-F1 {best['macro_f1']:.4f} at epoch {best['epoch']}")
    return model, best, hist

## 6. Smoke test - stops here while SMOKE is True

In [14]:
# The whole path on a few images before committing the GPU budget: real checkpoint, one
# update, validation, bundle save, reload, and inference from the reloaded bundle.
sm = {}
_m = build_model(); set_stage(_m, 'probe')
_bn = {k: v.clone() for k, v in _m.backbone.state_dict().items()
       if k.endswith('running_mean') or k.endswith('running_var')}
_w = _m.head.weight.detach().cpu().clone()
_opt = AdamW(param_groups(_m, PROBE_HEAD_LR, 0.0))
_sc = torch.amp.GradScaler('cuda', enabled=USE_SCALER)
_xb, _yb, _ = next(iter(loader(train_df.head(MICRO_BATCH), False, MICRO_BATCH)))
_m.train(); _m.backbone.eval()
with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
    _l = criterion(_m(_xb.to(device)), _yb.to(device))
(_sc.scale(_l) if _sc.is_enabled() else _l).backward()
for g in _opt.param_groups:
    for p in g['params']:
        if p.grad is not None:
            p.grad /= _yb.numel()
if _sc.is_enabled():
    _sc.unscale_(_opt); _sc.step(_opt); _sc.update()
else:
    _opt.step()
sm['loss_finite'] = bool(torch.isfinite(_l).item())
sm['head_moved'] = not torch.equal(_w, _m.head.weight.detach().cpu())
sm['probe_kept_batchnorm_frozen'] = all(
    torch.equal(_bn[k], v) for k, v in _m.backbone.state_dict().items() if k in _bn)
_p, _y, _, _ = evaluate(_m, val_df.head(8))
sm['validation_ran'] = {'rows': int(len(_y)),
                        'probs_sum_1': bool(np.allclose(_p.sum(1), 1, atol=1e-4))}

BUNDLE_FORMAT = 'mammofm-classical-bundle/1'
def save_bundle(path, model, temperature, metrics_, training_):
    b = {'format': BUNDLE_FORMAT,
         'architecture': {'encoder': 'efficientnet-b5', 'feat_dim': int(FEAT_DIM),
                          'source': 'Mammo-FM (batmanlab/Mammo-FM) image encoder',
                          'head': 'AdaptiveAvgPool2d(1) -> Linear(feat_dim, 3)',
                          'metadata_inputs': 'none - image only'},
         'class_mapping': {c: i for i, c in enumerate(CLASSES)},
         'class_names': list(CLASSES),
         'preprocessing': {'contract': PREP.CONTRACT, 'module_sha256': PREP_SHA},
         'temperature': None if temperature is None else float(temperature),
         'calibrated': temperature is not None,
         'metrics': metrics_, 'training': training_, 'versions': PKG,
         'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
         'intended_use': 'Research demonstration. Not validated for screening use and '
                         'not a clinical device.'}
    torch.save(b, path)
    return {k: v for k, v in b.items() if k != 'state_dict'}

os.makedirs(f'{PKG_DIR}/models', exist_ok=True)
_bp = f'{LOCAL}/_smoke_bundle.pth'
save_bundle(_bp, _m.cpu(), None, {'note': 'smoke'}, {'smoke': True})
_m.to(device)
_b = torch.load(_bp, map_location='cpu', weights_only=False)
assert _b['preprocessing']['module_sha256'] == PREP_SHA
_r = build_model().cpu(); _r.load_state_dict(_b['state_dict']); _r.eval()
_x1, _ = PREP.prepare_baseline(open(train_df.path.iloc[0], 'rb').read())
with torch.no_grad():
    _a = _r(torch.from_numpy(_x1).unsqueeze(0)).numpy()
    _c = _m.cpu()(torch.from_numpy(_x1).unsqueeze(0)).numpy()
_m.to(device)
sm['reloaded_matches_live'] = bool(np.allclose(_a, _c, atol=1e-5))
sm['max_logit_diff'] = float(np.abs(_a - _c).max())
print(json.dumps(sm, indent=2, default=str))
for k in ('loss_finite', 'head_moved', 'probe_kept_batchnorm_frozen',
          'reloaded_matches_live'):
    assert sm[k], f'SMOKE FAILED: {k}'
del _m, _r, _opt, _sc
torch.cuda.empty_cache()
print('\nsmoke passed.')
if SMOKE:
    raise SystemExit('SMOKE finished in %.1f min. Set SMOKE = False and run all again '
                     'for the real run.' % ((time.time() - T_START) / 60))

eval:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "loss_finite": true,
  "head_moved": true,
  "probe_kept_batchnorm_frozen": true,
  "validation_ran": {
    "rows": 8,
    "probs_sum_1": true
  },
  "reloaded_matches_live": true,
  "max_logit_diff": 0.0
}

smoke passed.


## 7. Train: probe, then fine-tune

In [15]:
# The budget covers TRAINING, measured from here - preprocessing time has already been
# spent and should not eat into it.
T_TRAIN = time.time()
BUDGET = TIME_BUDGET_HOURS * 3600
print(f'setup took {(T_TRAIN - T_START)/60:.1f} min. Training budget {BUDGET/3600:.1f} h '
      f'from now: {BUDGET*PROBE_SHARE/60:.0f} min probe, the rest fine-tuning.')

model = build_model()
model, probe_best, probe_hist = train_stage(
    model, 'probe', PROBE_HEAD_LR, 0.0, 'probe', BUDGET * PROBE_SHARE)

# whatever the probe did not use rolls into fine-tuning
ft_budget = max(300.0, BUDGET - (time.time() - T_TRAIN))
print(f'probe used {(time.time()-T_TRAIN)/60:.0f} min; '
      f'{ft_budget/60:.0f} min left for fine-tuning')
model, ft_best, ft_hist = train_stage(
    model, 'finetune', FT_HEAD_LR, FT_ENCODER_LR, 'finetune', ft_budget)

SELECTED = 'finetune' if ft_best['macro_f1'] >= probe_best['macro_f1'] else 'probe'
best = ft_best if SELECTED == 'finetune' else probe_best
model = build_model()
model.load_state_dict(torch.load(f'{RUN_DIR}/{SELECTED}_best.pth', map_location='cpu',
                                 weights_only=False)['model'])
model.to(device).eval()
print(f"selected {SELECTED}: probe {probe_best['macro_f1']:.4f} vs finetune "
      f"{ft_best['macro_f1']:.4f}, chosen on VALIDATION macro-F1 only")

setup took 4.1 min. Training budget 4.0 h from now: 84 min probe, the rest fine-tuning.
[probe] 6,147 trainable | 364 steps/epoch | budget 84 min


[probe] epoch 1:   0%|          | 0/364 [00:00<?, ?it/s]

eval:   0%|          | 0/32 [00:00<?, ?it/s]

[probe] epoch 1  2.0 min  val macro-F1 0.4426  acc 0.5114  mal-AUC 0.7142313047464709  | 82 min budget left, ~40 more epochs


[probe] epoch 2:   0%|          | 0/364 [00:00<?, ?it/s]

...

eval:   0%|          | 0/32 [00:00<?, ?it/s]

[finetune] epoch 18  4.0 min  val macro-F1 0.6533  acc 0.6635  mal-AUC 0.8351867848738334  | 154 min budget left, ~38 more epochs


[finetune] epoch 19:   0%|          | 0/364 [00:00<?, ?it/s]

eval:   0%|          | 0/32 [00:00<?, ?it/s]

[finetune] epoch 19  4.0 min  val macro-F1 0.6501  acc 0.6606  mal-AUC 0.836114678435333  | 150 min budget left, ~37 more epochs
[finetune] early stop: 5 epochs without improvement
[finetune] best val macro-F1 0.6536 at epoch 14
selected finetune: probe 0.5640 vs finetune 0.6536, chosen on VALIDATION macro-F1 only


## 8. Calibrate, then score the test partition once

In [16]:
# Temperature on the reserved calibration patients only.
TEMPERATURE = None
if len(calib_df) >= 50:
    _, y_c, _, lg_c = evaluate(model, calib_df)
    lt = torch.zeros(1, requires_grad=True)
    lg, yt = torch.tensor(lg_c), torch.tensor(y_c, dtype=torch.long)
    o = torch.optim.LBFGS([lt], lr=0.1, max_iter=60)
    def _cl():
        o.zero_grad()
        l = nn.functional.cross_entropy(lg / lt.exp(), yt)
        l.backward()
        return l
    o.step(_cl)
    TEMPERATURE = float(lt.exp().item())
    def _ece(p, y, bins=15):
        conf, pred = p.max(1), p.argmax(1)
        acc, e = (pred == y).astype(float), 0.0
        edges = np.linspace(0, 1, bins + 1)
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.any():
                e += m.mean() * abs(acc[m].mean() - conf[m].mean())
        return float(e)
    CALIB = {'temperature': TEMPERATURE, 'rows': int(len(y_c)),
             'patients': int(calib_df.patient.nunique()),
             'ece_before': _ece(torch.softmax(lg, 1).numpy(), y_c),
             'ece_after': _ece(torch.softmax(lg / TEMPERATURE, 1).numpy(), y_c),
             'note': 'both ECE values on the SAME calibration rows'}
else:
    CALIB = {'temperature': None, 'rows': int(len(calib_df)),
             'note': 'too few reserved calibration rows. Scores are UNCALIBRATED raw '
                     'softmax outputs and must be described that way.'}
print(json.dumps(CALIB, indent=2))

# Test, scored once, after selection and calibration.
probs_t, y_t, idx_t, lg_t = evaluate(model, test_df)
if TEMPERATURE:
    probs_t = torch.softmax(torch.tensor(lg_t) / TEMPERATURE, 1).numpy()
TEST, VAL = metrics(probs_t, y_t), metrics(*evaluate(model, val_df)[:2])

print(f"\nTEST  accuracy {TEST['accuracy']:.4f} | macro-F1 {TEST['macro_f1']:.4f}")
print(f"  malignant vs rest: ROC-AUC {TEST['malignant_vs_rest']['roc_auc']} | "
      f"AP {TEST['malignant_vs_rest']['average_precision']}")
for c, v in TEST['per_class'].items():
    print(f"    {c:<10s} P {v['precision']:.3f}  R {v['recall']:.3f}  "
          f"F1 {v['f1']:.3f}  n={v['support']}")
print('  confusion (rows true, cols predicted, Normal/Benign/Malignant):')
for row in TEST['confusion_matrix']:
    print('   ', row)

prior = train_df.groupby('source_dataset').label.agg(lambda s: s.value_counts().idxmax())
base_pred = test_df.source_dataset.map(prior).fillna(0).astype(int).to_numpy()
SOURCE_FLOOR = float(f1_score(test_df.label.astype(int).to_numpy(), base_pred,
                              average='macro', zero_division=0))
print(f'\nsource-name-only baseline macro-F1: {SOURCE_FLOOR:.4f}')
print(f"this model: {TEST['macro_f1']:.4f}  ->  " +
      ('ABOVE the floor' if TEST['macro_f1'] > SOURCE_FLOOR else
       'AT OR BELOW the floor - the model has not been shown to have learned anything '
       'about the images beyond which collection they came from.'))

pred_df = test_df.iloc[idx_t][['image_id', 'patient', 'source_dataset', 'role',
                               'label_text']].copy()
pred_df['label'] = y_t; pred_df['pred'] = probs_t.argmax(1)
pred_df['pred_name'] = [CLASSES[i] for i in pred_df.pred]
for i, c in enumerate(CLASSES):
    pred_df[f'p_{c}'] = probs_t[:, i]
pred_df.to_csv(f'{RUN_DIR}/test_predictions.csv', index=False)

SUMMARY = {
    'run': {'seed': SEED, 'selected_stage': SELECTED, 'smoke': SMOKE,
            'target': [TARGET_H, TARGET_W], 'micro_batch': MICRO_BATCH,
            'effective_batch': MICRO_BATCH * ACCUM,
            'wall_clock_hours': round((time.time() - T_START) / 3600, 2),
            'time_limited': True, 'when': time.strftime('%Y-%m-%d %H:%M:%S')},
    'data': {k: int(v) for k, v in
             {'train': len(train_df), 'val': len(val_df), 'calibration': len(calib_df),
              'test': len(test_df), 'excluded': len(EXCLUSIONS),
              'duplicate_groups': dupe_groups}.items()},
    'by_source': pd.crosstab(data.source_dataset, data.role).to_dict(),
    'training': {'probe': probe_best, 'finetune': ft_best,
                 'probe_history': probe_hist, 'finetune_history': ft_hist},
    'calibration': CALIB, 'validation': VAL, 'test': TEST,
    'source_name_only_floor_macro_f1': SOURCE_FLOOR,
    'encoder': {'sha256': FM_SHA[:32], 'feat_dim': FEAT_DIM},
    'preprocessing_sha256': PREP_SHA, 'versions': PKG,
    'intended_use': 'Research demonstration on Mammo-Bench. Trained under a fixed time '
                    'budget, not to convergence. Every partition comes from the same '
                    'collection, so this has NOT established external screening '
                    'performance.'}
json.dump(SUMMARY, open(f'{RUN_DIR}/summary.json', 'w'), indent=2, default=str)
print(f'\nsummary -> {RUN_DIR}/summary.json')

eval:   0%|          | 0/15 [00:00<?, ?it/s]

{
  "temperature": 1.3663203716278076,
  "rows": 920,
  "patients": 255,
  "ece_before": 0.05252901467940081,
  "ece_after": 0.020050775680852975,
  "note": "both ECE values on the SAME calibration rows"
}


eval:   0%|          | 0/46 [00:00<?, ?it/s]

eval:   0%|          | 0/32 [00:00<?, ?it/s]


TEST  accuracy 0.6736 | macro-F1 0.6598
  malignant vs rest: ROC-AUC 0.8417061017934221 | AP 0.7916078753272262
    Normal     P 0.671  R 0.807  F1 0.733  n=772
    Benign     P 0.587  R 0.454  F1 0.512  n=901
    Malignant  P 0.721  R 0.749  F1 0.735  n=1265
  confusion (rows true, cols predicted, Normal/Benign/Malignant):
    [623, 99, 50]
    [176, 409, 316]
    [129, 189, 947]

source-name-only baseline macro-F1: 0.5575
this model: 0.6598  ->  ABOVE the floor

summary -> /content/drive/MyDrive/mamobench-dataset/runs_demo_v1/full/seed_42/summary.json


In [18]:
# Recovery for the SameFileError in the last cell. Everything before it succeeded -
# the bundle is written and verified. This finishes the copy and prints the handover.
import os, json, shutil, time

copied, skipped = [], []
for src in (BUNDLE, PREP_SRC, f'{REPORTS}/exclusion_manifest.csv',
            f'{REPORTS}/manifest.csv'):
    if not os.path.exists(src):
        continue
    dst = f'{RUN_DIR}/{os.path.basename(src)}'
    if os.path.abspath(src) == os.path.abspath(dst):
        skipped.append(os.path.basename(src))       # already in place - this was the bug
        continue
    shutil.copy(src, dst)
    copied.append(os.path.basename(src))

json.dump(SUMMARY, open(f'{RUN_DIR}/evaluation_summary.json', 'w'), indent=2, default=str)
print('copied :', copied)
print('already in place :', skipped)
print('\nRUN_DIR contents:')
for f in sorted(os.listdir(RUN_DIR)):
    print(f'   {f}  ({os.path.getsize(f"{RUN_DIR}/{f}")/1e6:.1f} MB)')

print(f"""
================= COPY INTO THE REPO =================
  {RUN_DIR}/model_bundle.pth
      -> backend/routers/classical_session_analysis/models/model_bundle.pth
  {RUN_DIR}/preprocessing.py
      -> backend/routers/classical_session_analysis/preprocessing.py

  run.py, model.py, explain.py, router.py, selftest.py, __init__.py come from the
  handover ZIP. preprocessing.py MUST be this one - the bundle records its sha256
  ({PREP_SHA[:16]}) and load_bundle refuses a mismatch.

  Register in main.py:
      from routers.classical_session_analysis.router import router as classical_router
      app.include_router(classical_router)

  Then from the backend root:
      python -m routers.classical_session_analysis.selftest --integration

================= NUMBERS FOR THE DEMO =================
  test accuracy            {TEST['accuracy']:.4f}
  test macro-F1            {TEST['macro_f1']:.4f}
  source-name-only floor   {SOURCE_FLOOR:.4f}   <- quote this alongside, always
  malignant vs rest AUC    {TEST['malignant_vs_rest']['roc_auc']}
  malignant vs rest AP     {TEST['malignant_vs_rest']['average_precision']}
  selected stage           {SELECTED}
  calibration              {'temperature ' + format(TEMPERATURE, '.4f') if TEMPERATURE else 'NOT FITTED - scores are uncalibrated'}
  input size               {TARGET_H}x{TARGET_W}
  wall clock               {(time.time()-T_START)/3600:.2f} h, fixed budget, not to convergence
""")
for c, v in TEST['per_class'].items():
    print(f"  {c:<10s} precision {v['precision']:.3f}  recall {v['recall']:.3f}  "
          f"F1 {v['f1']:.3f}  n={v['support']}")
print('\n  confusion matrix (rows true, cols predicted, Normal/Benign/Malignant):')
for row in TEST['confusion_matrix']:
    print('   ', row)


copied : ['model_bundle.pth', 'preprocessing.py', 'exclusion_manifest.csv', 'manifest.csv']
already in place : []

RUN_DIR contents:
   evaluation_summary.json  (0.0 MB)
   exclusion_manifest.csv  (0.0 MB)
   finetune_best.pth  (122.6 MB)
   finetune_history.json  (0.0 MB)
   finetune_last.pth  (122.6 MB)
   manifest.csv  (11.4 MB)
   model_bundle.pth  (122.5 MB)
   preprocessing.py  (0.0 MB)
   probe_best.pth  (122.6 MB)
   probe_history.json  (0.0 MB)
   probe_last.pth  (122.6 MB)
   summary.json  (0.0 MB)
   test_predictions.csv  (0.3 MB)

================= COPY INTO THE REPO =================
  /content/drive/MyDrive/mamobench-dataset/runs_demo_v1/full/seed_42/model_bundle.pth
      -> backend/routers/classical_session_analysis/models/model_bundle.pth
  /content/drive/MyDrive/mamobench-dataset/runs_demo_v1/full/seed_42/preprocessing.py
      -> backend/routers/classical_session_analysis/preprocessing.py

  run.py, model.py, explain.py, router.py, selftest.py, __init__.py come from 

## 9. Bundle and handover

In [17]:
BUNDLE = f'{PKG_DIR}/models/model_bundle.pth'
meta = save_bundle(BUNDLE, model.cpu(), TEMPERATURE,
                   {'validation': VAL, 'test': TEST,
                    'source_name_only_floor_macro_f1': SOURCE_FLOOR},
                   {'selected_stage': SELECTED, 'seed': SEED,
                    'target': [TARGET_H, TARGET_W], 'micro_batch': MICRO_BATCH,
                    'effective_batch': MICRO_BATCH * ACCUM,
                    'time_limited': True, 'probe_head_lr': PROBE_HEAD_LR,
                    'ft_head_lr': FT_HEAD_LR, 'ft_encoder_lr': FT_ENCODER_LR,
                    'weight_decay': WEIGHT_DECAY, 'grad_clip': GRAD_CLIP})
model.to(device)
print(f"bundle {os.path.getsize(BUNDLE)/1e6:.0f} MB | calibrated={meta['calibrated']} "
      f"temperature={meta['temperature']}")

# reload it and confirm it predicts identically to the live model
chk = torch.load(BUNDLE, map_location='cpu', weights_only=False)
assert chk['preprocessing']['module_sha256'] == PREP_SHA
assert chk['class_names'] == CLASSES and chk['format'] == BUNDLE_FORMAT
rl = build_model().cpu(); rl.load_state_dict(chk['state_dict']); rl.eval()
xs = [PREP.prepare_baseline(open(p, 'rb').read())[0] for p in test_df.path.head(4)]
with torch.no_grad():
    a = rl(torch.from_numpy(np.stack(xs))).numpy()
    b = model.cpu()(torch.from_numpy(np.stack(xs))).numpy()
model.to(device)
assert np.allclose(a, b, atol=1e-5), np.abs(a - b).max()
print(f'reloaded bundle matches the live model (max logit diff {np.abs(a-b).max():.2e})')

# what the backend will actually return, built here so it can be seen before the demo
sm_probs = torch.softmax(torch.tensor(a) / (TEMPERATURE or 1.0), 1).numpy()
example = {CLASSES[i]: round(float(sm_probs[0][i]), 4) for i in range(3)}
print(f'example view output: result={CLASSES[int(sm_probs[0].argmax())]} '
      f'score={sm_probs[0].max():.4f} probabilities={example}')

# only small files go to Drive
for src in (BUNDLE, PREP_SRC, f'{RUN_DIR}/summary.json',
            f'{REPORTS}/exclusion_manifest.csv', f'{RUN_DIR}/test_predictions.csv'):
    if os.path.exists(src):
        shutil.copy(src, f'{RUN_DIR}/{os.path.basename(src)}')
json.dump(SUMMARY, open(f'{RUN_DIR}/evaluation_summary.json', 'w'), indent=2, default=str)
print(f'\nsaved to Drive: {RUN_DIR}')
print(sorted(os.listdir(RUN_DIR)))

print(f"""
COPY INTO THE REPO
  {BUNDLE}
      -> backend/routers/classical_session_analysis/models/model_bundle.pth
  {PREP_SRC}
      -> backend/routers/classical_session_analysis/preprocessing.py

  run.py, model.py, explain.py, router.py, selftest.py come from the handover ZIP and do
  not change between runs. preprocessing.py MUST be this one: the bundle records its
  sha256 ({PREP_SHA[:16]}) and load_bundle refuses a mismatch.

  Then, from the backend root:
      python -m routers.classical_session_analysis.selftest --integration

HEADLINE NUMBERS FOR THE DEMO
  test macro-F1            {TEST['macro_f1']:.4f}
  source-name-only floor   {SOURCE_FLOOR:.4f}
  malignant vs rest AUC    {TEST['malignant_vs_rest']['roc_auc']}
  trained for              {(time.time()-T_START)/3600:.2f} h under a fixed budget, not
                           to convergence
  input                    {TARGET_H}x{TARGET_W}, scores {'temperature-scaled' if TEMPERATURE else 'UNCALIBRATED'}
""")

bundle 123 MB | calibrated=True temperature=1.3663203716278076
reloaded bundle matches the live model (max logit diff 0.00e+00)
example view output: result=Benign score=0.6207 probabilities={'Normal': 0.0122, 'Benign': 0.6207, 'Malignant': 0.3671}


SameFileError: '/content/drive/MyDrive/mamobench-dataset/runs_demo_v1/full/seed_42/summary.json' and '/content/drive/MyDrive/mamobench-dataset/runs_demo_v1/full/seed_42/summary.json' are the same file

## Reading the result

This is a **research demonstration**, trained under a fixed time budget rather than to
convergence, on a single collection. It has not established external screening
performance and nothing here was validated on an external cohort.

Section 8 prints the macro-F1 obtainable from the dataset name alone. Source dataset is
strongly associated with the label in Mammo-Bench, so a model that has not cleared that
floor has not been shown to have learned anything about the images. Quote both numbers
together, not the model's alone.

If calibration could not be fitted, the scores are raw softmax outputs. They are still
ranked correctly, but they are not probabilities anyone should read as confidence.